# Uplift Modeling

This notebook is the central modeling notebook for the CRITEO-UPLIFTv2.1 causal
uplift study. It reads the frozen `f0`-`f11` feature contract, the sealed
70/15/15 train/validation/held-out split, and the frozen no-op preprocessing
transform established by the prior notebook, and evaluates uplift-ranking
methods through the frozen metric contract only.

Sections are added as each method's real, executed results exist -- this is a
running record, not a template with placeholder results. As of this run, T07
(Random reference + Response LightGBM baseline) is in progress: its
correctness/artifact-mechanism SMOKE rehearsal (D30) has executed; its FULL,
authoritative development run has not yet been authorized.


In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NOTEBOOK_PATH = REPO_ROOT / 'kaggle' / '02_uplift_modeling.ipynb'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
T04_CONFIG_PATH = REPO_ROOT / 'configs' / 't04_preprocessing.json'
T07_CONFIG_PATH = REPO_ROOT / 'configs' / 't07_baselines.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


## 0. Environment and reproducibility setup

In [2]:
import importlib

import lightgbm as lgb

from src.lightgbm_baseline import FROZEN_LIGHTGBM_VERSION

if lgb.__version__ != FROZEN_LIGHTGBM_VERSION:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'lightgbm=={FROZEN_LIGHTGBM_VERSION}'])
    raise RuntimeError(
        f'LightGBM was {lgb.__version__}; installed {FROZEN_LIGHTGBM_VERSION}. '
        'A hot importlib.reload() is not sufficient reproducibility enforcement for an '
        'already-imported compiled extension module -- restart the kernel and re-run this '
        'notebook from the top so the freshly-installed binary is what actually loads.'
    )

assert lgb.__version__ == FROZEN_LIGHTGBM_VERSION, (
    f'LightGBM version mismatch after guard: runtime={lgb.__version__}, frozen={FROZEN_LIGHTGBM_VERSION}. '
    'Refusing to fit on an unverified version.'
)
print(f'LightGBM verified: {lgb.__version__}')


LightGBM verified: 4.7.0


In [3]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import psutil
import sklearn
from sklearn.model_selection import train_test_split

from src.data import (
    DataContractError, FEATURE_COLUMNS, PROCESSED_COLUMNS, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME,
    assert_model_feature_contract, finalize_artifact_manifest, load_selector,
    materialize_pandas, open_processed_dataset, sha256_file, write_bytes_new,
    write_json_new, write_text_new,
)
from src.split import SplitDataset, SplitContractError, membership_hash
from src.preprocessing import IdentityFeatureTransform, preprocessing_contract
import src.metrics as metrics
import src.lightgbm_baseline as lgb_baseline

process = psutil.Process()
notebook_baseline_rss_bytes = process.memory_info().rss
print('Setup complete.')


Setup complete.


## 1. Modeling Objective

Two non-causal reference points are established before any causal estimator is
trained (D11, D12, D27): a **Random reference** (a seeded ranking independent
of `X`, `T`, and `Y`, giving the no-skill floor every method is compared
against), and a **Response LightGBM baseline** (`P(Y=1|X)`, a plain
factual-outcome classifier that never sees `T`).

Response probability and treatment effect are different quantities. A
"sure thing" -- someone who converts whether or not they are treated -- ranks
high on response but contributes zero incremental value; a "persuadable" --
who converts only if treated -- can rank low on response despite being exactly
who targeting should reach. A response model with strong AUC can therefore
still rank poorly on uplift, and that is the expected, reportable finding, not
an error to fix. Response's own diagnostics (ROC-AUC, average precision, log
loss) are evaluated separately from its uplift-ranking performance and never
select a causal winner (D27).

Both methods' scores are routed through the same frozen T06 metric interface
(`src/metrics.py`) used by every later estimator, so comparisons are on equal
footing from the start.


## 2. Data, Split & Preprocessing Contracts

In [4]:
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))
t04_config = json.loads(T04_CONFIG_PATH.read_text(encoding='utf-8'))
t07_config = json.loads(T07_CONFIG_PATH.read_text(encoding='utf-8'))

assert t05_config['lifecycle_state'] == 'T05_SPLIT_ACCEPTED', t05_config['lifecycle_state']
assert t04_config['lifecycle_state'] == 'T04_ACCEPTED', t04_config['lifecycle_state']

t05_run_manifest_path = REPO_ROOT / t05_config['lifecycle_state_evidence']['authorizing_run_manifest']
t05_run_root = t05_run_manifest_path.parent.parent
membership = pd.read_csv(t05_run_root / 'audit' / 'split_membership.csv')
observed_membership_hash = membership_hash(membership)
expected_membership_hash = t05_config['lifecycle_state_evidence']['membership_sha256']
if observed_membership_hash != expected_membership_hash:
    raise SplitContractError(
        f'Split membership hash mismatch: expected {expected_membership_hash}, observed {observed_membership_hash}'
    )

split_dataset = SplitDataset(membership=membership)
train_ids_full = split_dataset.train_ids()
validation_ids_full = split_dataset.validation_ids()
print(f'Split membership verified. train={len(train_ids_full):,} validation={len(validation_ids_full):,}')


Split membership verified. train=9,785,714 validation=2,096,938


In [5]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)
processed_sha256 = selector.payload['processed_sha256']

full_frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(full_frame.columns) != PROCESSED_COLUMNS:
    raise RuntimeError('Processed frame column order drifted from the frozen contract')

print(f'Processed dataset loaded: {len(full_frame):,} rows, sha256={processed_sha256[:16]}...')


Processed dataset loaded: 13,979,592 rows, sha256=fd2739bb074a50fa...


## 3. Scale-Gating (D30): SMOKE Verification

Under D30, T07 uses `SMOKE -> FULL` with `resource_gates = 0` (recorded and
justified in `configs/t07_baselines.json`: the full-scale data path is already
proven by T01, and a single Response LightGBM binary classifier over 12
numeric features does not meet D30's resource-risk trigger). SMOKE is a
bounded, development-only rehearsal of correctness, the feature/leakage
contract, row alignment, serialization/reload, and artifact mechanics -- it
never supports a performance claim or selects a model/config/seed. This
section executes SMOKE only; FULL is a separate, later-authorized run.


In [6]:
SMOKE_SIZE = t07_config['scale_gating']['smoke_size']
SMOKE_SEED = t07_config['scale_gating']['smoke_seed']
RESOURCE_GATES = t07_config['scale_gating']['resource_gates']
RUN_FULL_STAGE = True  # FULL authorized this run under D30 (resource_gates=0), after the SMOKE PASS above.

smoke_started = datetime.now(timezone.utc)
smoke_wall_start = __import__('time').perf_counter()

RUN_ID = smoke_started.strftime('t07_smoke_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

print(f'RUN_ID = {RUN_ID}')
print(f'resource_gates = {RESOURCE_GATES}, smoke_size = {SMOKE_SIZE}, seed = {SMOKE_SEED}')


RUN_ID = t07_smoke_20260818T111252Z_867633
resource_gates = 0, smoke_size = 50000, seed = 42


In [7]:
def _joint_strata(frame, id_column, treatment_column, outcome_column):
    return frame[treatment_column].astype(str) + '_' + frame[outcome_column].astype(str)


def smoke_sample_partition(partition_ids, quota, full_frame, seed):
    # One deterministic joint-(T,Y)-stratified draw of `quota` rows from
    # `partition_ids`, reusing the same train_test_split stratification
    # mechanism already used by src/split.py's assign_split() -- no new
    # sampling algorithm, no reusable scale-rung module.
    subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(partition_ids)]
    strata = _joint_strata(subset, SOURCE_ROW_ID, TREATMENT_COLUMN, PRIMARY_OUTCOME)
    selected_ids, _ = train_test_split(
        subset[SOURCE_ROW_ID].to_numpy(), train_size=quota, random_state=seed, stratify=strata,
    )
    return np.sort(selected_ids)


p_train = len(train_ids_full) / (len(train_ids_full) + len(validation_ids_full))
train_quota = round(SMOKE_SIZE * p_train)
validation_quota = SMOKE_SIZE - train_quota

smoke_train_ids = smoke_sample_partition(train_ids_full, train_quota, full_frame, SMOKE_SEED)
smoke_validation_ids = smoke_sample_partition(validation_ids_full, validation_quota, full_frame, SMOKE_SEED)

# Held-out isolation is proved positively, by construction, from the two
# sanctioned development partitions only (`train_ids_full`/`validation_ids_full`,
# obtained solely via SplitDataset.train_ids()/.validation_ids()). This never
# reads split_membership.csv's held_out label, never calls
# SplitDataset.held_out_ids(), and never inspects any held-out row, ID,
# feature, label, or summary through any path -- the held-out partition is
# simply absent from every set these assertions reference.
smoke_total = len(smoke_train_ids) + len(smoke_validation_ids)
development_ids_full = set(train_ids_full) | set(validation_ids_full)
assert smoke_total == SMOKE_SIZE, f'SMOKE total {smoke_total} != {SMOKE_SIZE}'
assert set(smoke_train_ids).issubset(set(train_ids_full)), 'smoke_train_ids must be a subset of the frozen train partition'
assert set(smoke_validation_ids).issubset(set(validation_ids_full)), 'smoke_validation_ids must be a subset of the frozen validation partition'
assert set(smoke_train_ids).isdisjoint(set(smoke_validation_ids)), 'smoke_train_ids and smoke_validation_ids must be disjoint'
assert (set(smoke_train_ids) | set(smoke_validation_ids)).issubset(development_ids_full), (
    'every SMOKE-selected ID must belong to train_ids_full union validation_ids_full'
)

print(f'SMOKE: train={len(smoke_train_ids):,} validation={len(smoke_validation_ids):,} total={smoke_total:,}')
print('Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.')


SMOKE: train=41,176 validation=8,824 total=50,000
Held-out isolation: proved by construction from train_ids_full/validation_ids_full only; held-out never read.


In [8]:
def joint_ty_support(ids, full_frame):
    subset = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(ids)]
    counts = subset.groupby([TREATMENT_COLUMN, PRIMARY_OUTCOME], observed=True).size()
    counts = counts.reindex(pd.MultiIndex.from_product([[0, 1], [0, 1]], names=[TREATMENT_COLUMN, PRIMARY_OUTCOME]), fill_value=0)
    return {f'T={t},Y={y}': int(n) for (t, y), n in counts.items()}


smoke_train_support = joint_ty_support(smoke_train_ids, full_frame)
smoke_validation_support = joint_ty_support(smoke_validation_ids, full_frame)
smoke_all_cells_nonempty = all(n > 0 for n in {**smoke_train_support, **smoke_validation_support}.values())
print('SMOKE train (T,Y) support:', smoke_train_support)
print('SMOKE validation (T,Y) support:', smoke_validation_support)
print('All joint-(T,Y) cells non-empty:', smoke_all_cells_nonempty)

smoke_sample_ids_frame = pd.concat([
    pd.DataFrame({SOURCE_ROW_ID: smoke_train_ids, 'partition': 'train'}),
    pd.DataFrame({SOURCE_ROW_ID: smoke_validation_ids, 'partition': 'validation'}),
], ignore_index=True)
smoke_sample_ids_sha256 = hashlib.sha256(
    smoke_sample_ids_frame.sort_values(SOURCE_ROW_ID)[SOURCE_ROW_ID].to_numpy(dtype='<i8').tobytes()
).hexdigest()

write_bytes_new(RUN_ROOT, 'audit/smoke_sample_row_ids.parquet', smoke_sample_ids_frame.to_parquet(index=False))
write_json_new(RUN_ROOT, 'audit/smoke_sample_manifest.json', {
    'run_id': RUN_ID,
    'stage': 't07_smoke',
    'population': 'smoke_50000_rows',
    'smoke_size': SMOKE_SIZE,
    'smoke_seed': SMOKE_SEED,
    'train_quota': int(train_quota),
    'validation_quota': int(validation_quota),
    'train_count': int(len(smoke_train_ids)),
    'validation_count': int(len(smoke_validation_ids)),
    'total_count': int(smoke_total),
    'train_support': smoke_train_support,
    'validation_support': smoke_validation_support,
    'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
    'smoke_sample_row_ids_sha256': smoke_sample_ids_sha256,
    'held_out_isolation_method': (
        'guaranteed_by_construction_from_sanctioned_development_partitions: smoke_train_ids and '
        'smoke_validation_ids are proved subsets of train_ids_full/validation_ids_full (each obtained '
        'solely via SplitDataset.train_ids()/.validation_ids()), pairwise disjoint, and their union is a '
        'subset of train_ids_full union validation_ids_full -- held-out is never read via '
        'split_membership.csv\'s held_out label, SplitDataset.held_out_ids(), or any other path'
    ),
})
print('SMOKE sample identity persisted.')


SMOKE train (T,Y) support: {'T=0,Y=0': 6164, 'T=0,Y=1': 12, 'T=1,Y=0': 34892, 'T=1,Y=1': 108}
SMOKE validation (T,Y) support: {'T=0,Y=0': 1321, 'T=0,Y=1': 3, 'T=1,Y=0': 7477, 'T=1,Y=1': 23}
All joint-(T,Y) cells non-empty: True


SMOKE sample identity persisted.


In [9]:
smoke_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_train_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
smoke_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(smoke_validation_ids)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)

transform = IdentityFeatureTransform()
X_smoke_train = transform.fit_transform(smoke_train_frame)
X_smoke_validation = transform.transform(smoke_validation_frame)
assert_model_feature_contract(X_smoke_train.columns)
assert_model_feature_contract(X_smoke_validation.columns)

y_smoke_train = smoke_train_frame[PRIMARY_OUTCOME].astype('float64')
y_smoke_validation = smoke_validation_frame[PRIMARY_OUTCOME].astype('float64')
t_smoke_validation = smoke_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
y_smoke_validation_arr = y_smoke_validation.to_numpy()
source_row_id_smoke_validation = smoke_validation_frame[SOURCE_ROW_ID].to_numpy()

print(f'X_smoke_train: {X_smoke_train.shape}, X_smoke_validation: {X_smoke_validation.shape}')
print('Feature contract verified: X is exactly', tuple(X_smoke_train.columns))


X_smoke_train: (41176, 12), X_smoke_validation: (8824, 12)
Feature contract verified: X is exactly ('f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11')


### 3.1 Response LightGBM baseline (SMOKE)

In [10]:
response_model = lgb_baseline.fit_binary_classifier(
    X_smoke_train, y_smoke_train, X_smoke_validation, y_smoke_validation,
)
response_probabilities_smoke = lgb_baseline.predict_probabilities(response_model, X_smoke_validation)

assert np.isfinite(response_probabilities_smoke).all()
assert (response_probabilities_smoke >= 0).all() and (response_probabilities_smoke <= 1).all()
assert len(response_probabilities_smoke) == len(smoke_validation_ids)
assert set(source_row_id_smoke_validation) == set(smoke_validation_ids)

print(f'Response fit complete. best_iteration={response_model.best_iteration}, config_hash={response_model.config_hash[:16]}...')
print('Predictions bounded/finite/aligned: OK')


Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[19]	validation's binary_logloss: 0.0194955
Response fit complete. best_iteration=19, config_hash=647b625dca13ea63...
Predictions bounded/finite/aligned: OK


### 3.2 Random reference (SMOKE) -- via the public T06 interface only

In [11]:
random_scores_smoke = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
random_ranking_smoke = metrics.evaluate_ranking(
    random_scores_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
)
random_reference_distribution_smoke = metrics.random_ranking_reference_distribution(
    t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
)
assert len(random_reference_distribution_smoke) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

response_ranking_smoke = metrics.evaluate_ranking(
    response_probabilities_smoke, t_smoke_validation, y_smoke_validation_arr, source_row_id_smoke_validation,
)
response_diag_smoke = metrics.response_diagnostics(response_probabilities_smoke, y_smoke_validation_arr)
ate_smoke = metrics.compute_ate(t_smoke_validation, y_smoke_validation_arr)

print(f'Random: 1 illustrative draw + {len(random_reference_distribution_smoke)} reference draws computed via the public T06 interface.')
print(f'Response ranking qini_above_random (SMOKE, non-substantive): {response_ranking_smoke.qini_above_random:.6f}')


Random: 1 illustrative draw + 200 reference draws computed via the public T06 interface.
Response ranking qini_above_random (SMOKE, non-substantive): -1.875556


### 3.3 SMOKE artifacts

In [12]:
def _rows_with_run_context(rows, population='smoke_50000_rows'):
    for row in rows:
        row = dict(row)
        row.setdefault('run_id', RUN_ID)
        row.setdefault('stage', 't07_smoke')
        row.setdefault('population', population)
        yield row


ate_summary_rows = list(_rows_with_run_context([{'method': 'assigned_arm', **ate_smoke.__dict__}]))
response_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))

uplift_at_k_rows = []
for label in metrics.RANKING_K_LABELS:
    uplift_at_k_rows.append({'method': 'random', 'k': label, 'uplift': random_ranking_smoke.uplift_at_k[label],
                              'incremental_conversions': random_ranking_smoke.incremental_conversions_at_k[label],
                              'status': random_ranking_smoke.top_k_status[label]})
    uplift_at_k_rows.append({'method': 'response', 'k': label, 'uplift': response_ranking_smoke.uplift_at_k[label],
                              'incremental_conversions': response_ranking_smoke.incremental_conversions_at_k[label],
                              'status': response_ranking_smoke.top_k_status[label]})
uplift_at_k_rows = list(_rows_with_run_context(uplift_at_k_rows))

model_summary_rows = []
for name, result in (('random', random_ranking_smoke), ('response', response_ranking_smoke)):
    model_summary_rows.append({
        'ranking_method': name,
        'qini_area': result.qini_area,
        'theoretical_random_qini_area': result.theoretical_random_qini_area,
        'qini_above_random': result.qini_above_random,
        'qini_above_random_permutation': random_ranking_smoke.qini_above_random,
        'uplift_at_10pct': result.uplift_at_k['10pct'],
        'uplift_at_20pct': result.uplift_at_k['20pct'],
        'uplift_at_30pct': result.uplift_at_k['30pct'],
        'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
        'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
        'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
    })
model_summary_rows = list(_rows_with_run_context(model_summary_rows))

random_deciles_rows = list(_rows_with_run_context(random_ranking_smoke.decile_table.to_dict('records')))
response_deciles_rows = list(_rows_with_run_context(response_ranking_smoke.decile_table.to_dict('records')))

for name, rows in (
    ('tables/ate_summary.csv', ate_summary_rows),
    ('tables/response_diagnostics.csv', response_diagnostics_rows),
    ('tables/random_deciles.csv', random_deciles_rows),
    ('tables/response_deciles.csv', response_deciles_rows),
    ('tables/uplift_at_k.csv', uplift_at_k_rows),
    ('tables/model_summary.csv', model_summary_rows),
):
    write_text_new(RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

print('SMOKE tables written.')


SMOKE tables written.


In [13]:
response_predictions_frame = pd.DataFrame({
    SOURCE_ROW_ID: source_row_id_smoke_validation,
    'response_probability': response_probabilities_smoke,
})
random_scores_frame = pd.DataFrame({
    SOURCE_ROW_ID: source_row_id_smoke_validation,
    'random_score': random_scores_smoke,
})
response_predictions_bytes = response_predictions_frame.to_parquet(index=False)
random_scores_bytes = random_scores_frame.to_parquet(index=False)

write_bytes_new(RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes)
write_bytes_new(RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes)

response_model_text = response_model.booster.model_to_string()
write_text_new(RUN_ROOT, 'models/response_model.txt', response_model_text)

random_baseline_summary_rows = list(_rows_with_run_context([{
    'theoretical_random_qini_area': random_ranking_smoke.theoretical_random_qini_area,
    'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
    'illustrative_draw_qini_area': random_ranking_smoke.qini_area,
    'illustrative_draw_qini_above_random': random_ranking_smoke.qini_above_random,
}]))
write_text_new(RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows).to_csv(index=False, lineterminator='\n'))

random_baseline_draws_rows = list(_rows_with_run_context(random_reference_distribution_smoke.to_dict('records')))
write_text_new(RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows).to_csv(index=False, lineterminator='\n'))

model_probability_diagnostics_rows = list(_rows_with_run_context([{'method': 'response', **response_diag_smoke.__dict__}]))
write_text_new(RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows).to_csv(index=False, lineterminator='\n'))

definitions = metrics.metric_definitions()
definitions_sha256 = hashlib.sha256(json.dumps(definitions, sort_keys=True).encode()).hexdigest()

def _flatten(value):
    return value if isinstance(value, (str, int, float, bool)) or value is None else json.dumps(value)

definitions_rows = list(_rows_with_run_context(
    [{'field': k, 'value': _flatten(v)} for k, v in definitions.items()]
    + [{'field': 'definitions_sha256', 'value': definitions_sha256}]
))
write_text_new(RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows).to_csv(index=False, lineterminator='\n'))

response_predictions_sha256 = hashlib.sha256(response_predictions_bytes).hexdigest()
random_scores_sha256 = hashlib.sha256(random_scores_bytes).hexdigest()
print('SMOKE audit/model/prediction artifacts written.')


SMOKE audit/model/prediction artifacts written.


### 3.4 SMOKE reproducibility check (T07.7, SMOKE scope)

In [14]:
reloaded_booster = lgb.Booster(model_str=response_model_text)
reloaded_config_hash = lgb_baseline.config_hash()
config_hash_matches = reloaded_config_hash == response_model.config_hash

X_smoke_validation_rebuilt = transform.transform(smoke_validation_frame)
reloaded_probabilities = np.asarray(reloaded_booster.predict(X_smoke_validation_rebuilt, num_iteration=response_model.best_iteration), dtype=np.float64)
prediction_reload_matches = np.allclose(reloaded_probabilities, response_probabilities_smoke, rtol=1e-6, atol=1e-8)

row_identity_matches = set(source_row_id_smoke_validation) == set(smoke_validation_ids)

random_scores_regenerated = metrics.seeded_random_scores(len(smoke_validation_ids), seed=metrics.RANDOM_RANKING_SEED)
random_scores_exact_match = np.array_equal(random_scores_regenerated, random_scores_smoke)

reload_verification = {
    'config_hash_matches': bool(config_hash_matches),
    'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches),
    'row_identity_matches': bool(row_identity_matches),
    'random_scores_exact_match': bool(random_scores_exact_match),
    'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
}
print(reload_verification)
assert all(reload_verification[k] for k in ('config_hash_matches', 'prediction_reload_matches_within_tolerance', 'row_identity_matches', 'random_scores_exact_match'))


{'config_hash_matches': True, 'prediction_reload_matches_within_tolerance': True, 'row_identity_matches': True, 'random_scores_exact_match': True, 'tolerance': {'rtol': 1e-06, 'atol': 1e-08}}


In [15]:
smoke_wall_seconds = __import__('time').perf_counter() - smoke_wall_start
smoke_peak_rss_bytes = process.memory_info().rss

write_json_new(RUN_ROOT, 'audit/environment.json', {
    'run_id': RUN_ID,
    'created_at_utc': smoke_started.isoformat(),
    'git_head': git_head,
    'git_dirty': git_dirty,
    'python': sys.version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'lightgbm': lgb.__version__,
    'psutil': psutil.__version__,
    'total_ram_bytes': psutil.virtual_memory().total,
})

write_json_new(RUN_ROOT, 'audit/run_config.json', {
    'run_id': RUN_ID,
    'created_at_utc': smoke_started.isoformat(),
    'stage': 't07_smoke',
    'population': 'smoke_50000_rows',
    'git_head': git_head,
    'git_dirty': git_dirty,
    'real_or_held_out_data_accessed': False,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
    'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
    'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
    'processed_sha256': processed_sha256,
    'split_membership_sha256': observed_membership_hash,
    't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
    't04_lifecycle_state': t04_config['lifecycle_state'],
    't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
    't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
    'scale_gating': {'policy': 'D30', 'stage': 'SMOKE', 'resource_gates': RESOURCE_GATES, 'smoke_size': SMOKE_SIZE, 'smoke_seed': SMOKE_SEED},
    'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
    'lightgbm_config_hash': response_model.config_hash,
    'lightgbm_best_iteration': response_model.best_iteration,
    'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
    'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
    'response_predictions_sha256': response_predictions_sha256,
    'random_scores_sha256': random_scores_sha256,
    'definitions_sha256': definitions_sha256,
    'resource_evidence': {
        'wall_seconds': smoke_wall_seconds,
        'baseline_rss_bytes': notebook_baseline_rss_bytes,
        'peak_rss_bytes': smoke_peak_rss_bytes,
        'peak_rss_delta_bytes': smoke_peak_rss_bytes - notebook_baseline_rss_bytes,
    },
    'reload_verification': reload_verification,
})

print(f'SMOKE resource evidence: wall_seconds={smoke_wall_seconds:.2f}, peak_rss_delta_bytes={smoke_peak_rss_bytes - notebook_baseline_rss_bytes:,}')


SMOKE resource evidence: wall_seconds=109.31, peak_rss_delta_bytes=3,742,142,464


In [16]:
finalize_artifact_manifest(
    RUN_ROOT,
    run_id=RUN_ID,
    final_status='COMPLETED_T07_SMOKE_VERIFIED',
    created_at_utc=datetime.now(timezone.utc).isoformat(),
    stage='t07_smoke',
    population='smoke_50000_rows',
    external_artifacts=[
        {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
         'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
        {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
        {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
    ],
)
print(f'SMOKE run finalized: {RUN_ID}')

immutable_write_refused = False
try:
    write_json_new(RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
except (FileExistsError, DataContractError):
    immutable_write_refused = True
except Exception:
    immutable_write_refused = False
print(f'Immutable-run write refusal verified: {immutable_write_refused}')
assert immutable_write_refused


SMOKE run finalized: t07_smoke_20260818T111252Z_867633
Immutable-run write refusal verified: True


### 3.5 SMOKE summary

In [17]:
smoke_summary = {
    'run_id': RUN_ID,
    'smoke_total': int(smoke_total),
    'smoke_train_count': int(len(smoke_train_ids)),
    'smoke_validation_count': int(len(smoke_validation_ids)),
    'all_joint_ty_cells_nonempty': bool(smoke_all_cells_nonempty),
    'lightgbm_version': lgb.__version__,
    'lightgbm_config_hash': response_model.config_hash,
    'best_iteration': response_model.best_iteration,
    'random_reference_draws': len(random_reference_distribution_smoke),
    'reload_verification': reload_verification,
    'resource_wall_seconds': smoke_wall_seconds,
    'immutable_write_refused': immutable_write_refused,
}
print(json.dumps(smoke_summary, indent=2))


{
  "run_id": "t07_smoke_20260818T111252Z_867633",
  "smoke_total": 50000,
  "smoke_train_count": 41176,
  "smoke_validation_count": 8824,
  "all_joint_ty_cells_nonempty": true,
  "lightgbm_version": "4.7.0",
  "lightgbm_config_hash": "647b625dca13ea63f61a8ded8ce97c5e4250beea1d09cc6fa1e3ef3bbb6eaf30",
  "best_iteration": 19,
  "random_reference_draws": 200,
  "reload_verification": {
    "config_hash_matches": true,
    "prediction_reload_matches_within_tolerance": true,
    "row_identity_matches": true,
    "random_scores_exact_match": true,
    "tolerance": {
      "rtol": 1e-06,
      "atol": 1e-08
    }
  },
  "resource_wall_seconds": 109.31204030002118,
  "immutable_write_refused": true
}


## Scale-Gating (D30): FULL

SMOKE passed every correctness/artifact-mechanism check in the section above.
Under D30, T07's approved path is `SMOKE -> FULL` with `resource_gates = 0`
(recorded and justified in `configs/t07_baselines.json`), so FULL is the next
and only remaining stage. FULL fits on the **complete** frozen train partition,
early-stops on the **complete** frozen validation partition, and is the
authoritative development run whose results the sections below report.
Held-out remains completely sealed throughout -- this section never reads
`SplitDataset.held_out_ids()` or any held-out row.

In [18]:
from IPython.display import Markdown, display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

full_started = datetime.now(timezone.utc)
full_wall_start = __import__('time').perf_counter()
full_baseline_rss_bytes = process.memory_info().rss

FULL_RUN_ID = full_started.strftime('t07_full_%Y%m%dT%H%M%SZ_%f')
FULL_RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / FULL_RUN_ID
FULL_RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    full_git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    full_git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    full_git_head, full_git_dirty = None, None

print(f'FULL_RUN_ID = {FULL_RUN_ID}')


FULL_RUN_ID = t07_full_20260818T111446Z_593205


In [19]:
full_train_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(train_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
full_validation_frame = full_frame.loc[full_frame[SOURCE_ROW_ID].isin(validation_ids_full)].sort_values(SOURCE_ROW_ID).reset_index(drop=True)
assert len(full_train_frame) == len(train_ids_full)
assert len(full_validation_frame) == len(validation_ids_full)

transform_full = IdentityFeatureTransform()
X_full_train = transform_full.fit_transform(full_train_frame)
X_full_validation = transform_full.transform(full_validation_frame)
assert_model_feature_contract(X_full_train.columns)
assert_model_feature_contract(X_full_validation.columns)

y_full_train = full_train_frame[PRIMARY_OUTCOME].astype('float64')
y_full_validation = full_validation_frame[PRIMARY_OUTCOME].astype('float64')
t_full_validation = full_validation_frame[TREATMENT_COLUMN].astype('float64').to_numpy()
y_full_validation_arr = y_full_validation.to_numpy()
source_row_id_full_validation = full_validation_frame[SOURCE_ROW_ID].to_numpy()

print(f'FULL: X_train={X_full_train.shape}, X_validation={X_full_validation.shape}')


FULL: X_train=(9785714, 12), X_validation=(2096938, 12)


## 4. Random Reference

In [20]:
random_scores_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
random_ranking_full = metrics.evaluate_ranking(
    random_scores_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
)
random_reference_distribution_full = metrics.random_ranking_reference_distribution(
    t_full_validation, y_full_validation_arr, source_row_id_full_validation,
)
assert len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200

display(Markdown(
    f"The theoretical expected-random Qini area on the frozen validation population is "
    f"**{random_ranking_full.theoretical_random_qini_area:.4f}**. One deterministic seed-42 illustrative "
    f"random ranking realizes `qini_area = {random_ranking_full.qini_area:.4f}` "
    f"(`qini_above_random = {random_ranking_full.qini_above_random:.4f}`), and the frozen "
    f"{len(random_reference_distribution_full)}-draw random-ranking reference distribution has "
    f"`qini_above_random` mean **{random_reference_distribution_full['qini_above_random'].mean():.4f}** "
    f"(std {random_reference_distribution_full['qini_above_random'].std():.4f}), consistent with a "
    f"no-skill ranking centered near zero. This distribution is secondary empirical context; the "
    f"theoretical line remains the primary random reference (D11)."
))


The theoretical expected-random Qini area on the frozen validation population is **1024.6686**. One deterministic seed-42 illustrative random ranking realizes `qini_area = 1006.1794` (`qini_above_random = -18.4892`), and the frozen 200-draw random-ranking reference distribution has `qini_above_random` mean **-1.5886** (std 40.8457), consistent with a no-skill ranking centered near zero. This distribution is secondary empirical context; the theoretical line remains the primary random reference (D11).

## 5. Response LightGBM Baseline

In [21]:
response_model_full = lgb_baseline.fit_binary_classifier(
    X_full_train, y_full_train, X_full_validation, y_full_validation,
)
response_probabilities_full = lgb_baseline.predict_probabilities(response_model_full, X_full_validation)

assert np.isfinite(response_probabilities_full).all()
assert (response_probabilities_full >= 0).all() and (response_probabilities_full <= 1).all()
assert set(source_row_id_full_validation) == set(validation_ids_full)

response_diag_full = metrics.response_diagnostics(response_probabilities_full, y_full_validation_arr)
response_ranking_full = metrics.evaluate_ranking(
    response_probabilities_full, t_full_validation, y_full_validation_arr, source_row_id_full_validation,
)
ate_full = metrics.compute_ate(t_full_validation, y_full_validation_arr)

print(f'Response fit complete. best_iteration={response_model_full.best_iteration}, config_hash={response_model_full.config_hash[:16]}...')


Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[1]	validation's binary_logloss: 0.0207656


Response fit complete. best_iteration=1, config_hash=647b625dca13ea63...


In [22]:
display(Markdown(
    f"**Response diagnostics (factual-outcome prediction quality -- diagnostic only, D27; never a causal "
    f"ranking claim):** ROC-AUC = {response_diag_full.roc_auc:.4f}, average precision = "
    f"{response_diag_full.average_precision:.4f}, log loss = {response_diag_full.log_loss:.4f}.\n\n"
    f"**Response as an uplift ranking** (the same probability scores, evaluated as a policy via the "
    f"identical T06 interface used for every method): `qini_area = {response_ranking_full.qini_area:.4f}`, "
    f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` "
    f"(theoretical random = {response_ranking_full.theoretical_random_qini_area:.4f}). "
    f"`uplift@10% = {response_ranking_full.uplift_at_k['10pct']}`, "
    f"`uplift@20% = {response_ranking_full.uplift_at_k['20pct']}`, "
    f"`uplift@30% = {response_ranking_full.uplift_at_k['30pct']}` "
    f"(status: {response_ranking_full.top_k_status['10pct']}/{response_ranking_full.top_k_status['20pct']}/{response_ranking_full.top_k_status['30pct']}).\n\n"
    f"**Assigned-arm ATE** (population aggregate, not a ranking estimator; D24 methodology note): "
    f"{ate_full.ate:.4f} ({ate_full.ate_percentage_points:.2f} pp), 95% CI "
    f"[{ate_full.ci_95_low:.4f}, {ate_full.ci_95_high:.4f}]."
))


**Response diagnostics (factual-outcome prediction quality -- diagnostic only, D27; never a causal ranking claim):** ROC-AUC = 0.9038, average precision = 0.2033, log loss = 0.0208.

**Response as an uplift ranking** (the same probability scores, evaluated as a policy via the identical T06 interface used for every method): `qini_area = -1114.6439`, `qini_above_random = -2139.3125` (theoretical random = 1024.6686). `uplift@10% = 0.007330652775548674`, `uplift@20% = -0.006070526631827052`, `uplift@30% = -0.010081975358519447` (status: OK/OK/OK).

**Assigned-arm ATE** (population aggregate, not a ranking estimator; D24 methodology note): 0.0011 (0.11 pp), 95% CI [0.0010, 0.0013].

### FULL artifacts

In [23]:
def _full_rows_with_run_context(rows, population='full_development_population'):
    for row in rows:
        row = dict(row)
        row.setdefault('run_id', FULL_RUN_ID)
        row.setdefault('stage', 't07_full')
        row.setdefault('population', population)
        yield row


ate_summary_rows_full = list(_full_rows_with_run_context([{'method': 'assigned_arm', **ate_full.__dict__}]))
response_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))

uplift_at_k_rows_full = []
for label in metrics.RANKING_K_LABELS:
    uplift_at_k_rows_full.append({'method': 'random', 'k': label, 'uplift': random_ranking_full.uplift_at_k[label],
                                   'incremental_conversions': random_ranking_full.incremental_conversions_at_k[label],
                                   'status': random_ranking_full.top_k_status[label]})
    uplift_at_k_rows_full.append({'method': 'response', 'k': label, 'uplift': response_ranking_full.uplift_at_k[label],
                                   'incremental_conversions': response_ranking_full.incremental_conversions_at_k[label],
                                   'status': response_ranking_full.top_k_status[label]})
uplift_at_k_rows_full = list(_full_rows_with_run_context(uplift_at_k_rows_full))

model_summary_rows_full = []
for name, result in (('random', random_ranking_full), ('response', response_ranking_full)):
    model_summary_rows_full.append({
        'ranking_method': name,
        'qini_area': result.qini_area,
        'theoretical_random_qini_area': result.theoretical_random_qini_area,
        'qini_above_random': result.qini_above_random,
        'qini_above_random_permutation': random_ranking_full.qini_above_random,
        'uplift_at_10pct': result.uplift_at_k['10pct'],
        'uplift_at_20pct': result.uplift_at_k['20pct'],
        'uplift_at_30pct': result.uplift_at_k['30pct'],
        'incremental_conversions_at_10pct': result.incremental_conversions_at_k['10pct'],
        'incremental_conversions_at_20pct': result.incremental_conversions_at_k['20pct'],
        'incremental_conversions_at_30pct': result.incremental_conversions_at_k['30pct'],
    })
model_summary_rows_full = list(_full_rows_with_run_context(model_summary_rows_full))

random_deciles_rows_full = list(_full_rows_with_run_context(random_ranking_full.decile_table.to_dict('records')))
response_deciles_rows_full = list(_full_rows_with_run_context(response_ranking_full.decile_table.to_dict('records')))

for name, rows in (
    ('tables/ate_summary.csv', ate_summary_rows_full),
    ('tables/response_diagnostics.csv', response_diagnostics_rows_full),
    ('tables/random_deciles.csv', random_deciles_rows_full),
    ('tables/response_deciles.csv', response_deciles_rows_full),
    ('tables/uplift_at_k.csv', uplift_at_k_rows_full),
    ('tables/model_summary.csv', model_summary_rows_full),
):
    write_text_new(FULL_RUN_ROOT, name, pd.DataFrame(rows).to_csv(index=False, lineterminator='\n'))

print('FULL tables written.')


FULL tables written.


In [24]:
response_predictions_frame_full = pd.DataFrame({
    SOURCE_ROW_ID: source_row_id_full_validation,
    'response_probability': response_probabilities_full,
})
random_scores_frame_full = pd.DataFrame({
    SOURCE_ROW_ID: source_row_id_full_validation,
    'random_score': random_scores_full,
})
response_predictions_bytes_full = response_predictions_frame_full.to_parquet(index=False)
random_scores_bytes_full = random_scores_frame_full.to_parquet(index=False)

write_bytes_new(FULL_RUN_ROOT, 'predictions/development/response/seed_42/validation_predictions.parquet', response_predictions_bytes_full)
write_bytes_new(FULL_RUN_ROOT, 'predictions/development/random/seed_42/validation_scores.parquet', random_scores_bytes_full)

response_model_text_full = response_model_full.booster.model_to_string()
write_text_new(FULL_RUN_ROOT, 'models/response_model.txt', response_model_text_full)

random_baseline_summary_rows_full = list(_full_rows_with_run_context([{
    'theoretical_random_qini_area': random_ranking_full.theoretical_random_qini_area,
    'illustrative_draw_seed': metrics.RANDOM_RANKING_SEED,
    'illustrative_draw_qini_area': random_ranking_full.qini_area,
    'illustrative_draw_qini_above_random': random_ranking_full.qini_above_random,
}]))
write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_summary.csv', pd.DataFrame(random_baseline_summary_rows_full).to_csv(index=False, lineterminator='\n'))

random_baseline_draws_rows_full = list(_full_rows_with_run_context(random_reference_distribution_full.to_dict('records')))
write_text_new(FULL_RUN_ROOT, 'audit/random_baseline_draws.csv', pd.DataFrame(random_baseline_draws_rows_full).to_csv(index=False, lineterminator='\n'))

model_probability_diagnostics_rows_full = list(_full_rows_with_run_context([{'method': 'response', **response_diag_full.__dict__}]))
write_text_new(FULL_RUN_ROOT, 'audit/model_probability_diagnostics.csv', pd.DataFrame(model_probability_diagnostics_rows_full).to_csv(index=False, lineterminator='\n'))

definitions_full = metrics.metric_definitions()
definitions_sha256_full = hashlib.sha256(json.dumps(definitions_full, sort_keys=True).encode()).hexdigest()
definitions_rows_full = list(_full_rows_with_run_context(
    [{'field': k, 'value': _flatten(v)} for k, v in definitions_full.items()]
    + [{'field': 'definitions_sha256', 'value': definitions_sha256_full}]
))
write_text_new(FULL_RUN_ROOT, 'audit/metric_definitions.csv', pd.DataFrame(definitions_rows_full).to_csv(index=False, lineterminator='\n'))

response_predictions_sha256_full = hashlib.sha256(response_predictions_bytes_full).hexdigest()
random_scores_sha256_full = hashlib.sha256(random_scores_bytes_full).hexdigest()
print('FULL audit/model/prediction artifacts written.')


FULL audit/model/prediction artifacts written.


In [25]:
import io


def _cumulative_rate_curve(decile_table):
    ordered = decile_table.sort_values('decile')
    cum_n1 = ordered['n1'].cumsum()
    cum_n0 = ordered['n0'].cumsum()
    cum_y1 = ordered['y1'].cumsum()
    cum_y0 = ordered['y0'].cumsum()
    with np.errstate(divide='ignore', invalid='ignore'):
        cumulative_rate = (cum_y1 / cum_n1) - (cum_y0 / cum_n0)
    return ordered['decile'].to_numpy(), cumulative_rate.to_numpy()


def _save_figure_bytes(fig, run_root, relative_path):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    write_bytes_new(run_root, relative_path, buffer.getvalue())


fig, ax = plt.subplots(figsize=(7, 4))
deciles = response_ranking_full.decile_table.sort_values('decile')['decile']
ax.bar(deciles - 0.15, response_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Response')
ax.bar(deciles + 0.15, random_ranking_full.decile_table.sort_values('decile')['observed_uplift'], width=0.3, label='Random')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Decile (1 = highest-ranked)')
ax.set_ylabel('Observed uplift (treated_rate - control_rate)')
ax.set_title('Response vs. Random: observed uplift by decile')
ax.legend()
fig.tight_layout()
_save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/response_uplift_deciles.png')

fig, ax = plt.subplots(figsize=(7, 4))
resp_d, resp_rate = _cumulative_rate_curve(response_ranking_full.decile_table)
rand_d, rand_rate = _cumulative_rate_curve(random_ranking_full.decile_table)
ax.plot(resp_d, resp_rate, marker='o', label='Response')
ax.plot(rand_d, rand_rate, marker='o', label='Random')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Cumulative decile coverage')
ax.set_ylabel('Cumulative uplift rate')
ax.set_title('Cumulative uplift rate by coverage')
ax.legend()
fig.tight_layout()
_save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_rate.png')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
ax.set_xlabel('Coverage')
ax.set_ylabel('Qini gain (incremental conversions)')
ax.set_title('Cumulative Qini gain by coverage')
ax.legend()
fig.tight_layout()
_save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/cumulative_uplift_gain.png')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(response_ranking_full.qini_curve['coverage'], response_ranking_full.qini_curve['qini_gain'], label='Response')
ax.plot(random_ranking_full.qini_curve['coverage'], random_ranking_full.qini_curve['qini_gain'], label='Random (one draw)')
theoretical_x = np.array([0.0, 1.0])
theoretical_y = theoretical_x * (response_ranking_full.qini_curve['qini_gain'].iloc[-1])
ax.plot(theoretical_x, theoretical_y, linestyle='--', color='gray', label='Theoretical random line')
ax.set_xlabel('Coverage')
ax.set_ylabel('Qini gain')
ax.set_title('Qini curve: Response vs. Random vs. theoretical random')
ax.legend()
fig.tight_layout()
_save_figure_bytes(fig, FULL_RUN_ROOT, 'figures/qini_curve.png')

print('FULL figures written.')


FULL figures written.


### FULL reproducibility check (T07.7)

In [26]:
reloaded_booster_full = lgb.Booster(model_str=response_model_text_full)
reloaded_config_hash_full = lgb_baseline.config_hash()
config_hash_matches_full = reloaded_config_hash_full == response_model_full.config_hash

X_full_validation_rebuilt = transform_full.transform(full_validation_frame)
reloaded_probabilities_full = np.asarray(
    reloaded_booster_full.predict(X_full_validation_rebuilt, num_iteration=response_model_full.best_iteration), dtype=np.float64
)
prediction_reload_matches_full = np.allclose(reloaded_probabilities_full, response_probabilities_full, rtol=1e-6, atol=1e-8)
row_identity_matches_full = set(source_row_id_full_validation) == set(validation_ids_full)

random_scores_regenerated_full = metrics.seeded_random_scores(len(validation_ids_full), seed=metrics.RANDOM_RANKING_SEED)
random_scores_exact_match_full = np.array_equal(random_scores_regenerated_full, random_scores_full)

# The frozen 200-draw distribution is not recomputed a second time to "prove" determinism here --
# T06's own test suite (test_random_ranking_reference_is_deterministic_given_master_seed) already
# covers that. This is a bounded self-consistency check of what was actually stored above.
random_reference_self_consistent_full = (
    len(random_reference_distribution_full) == metrics.RANDOM_RANKING_REFERENCE_DRAWS == 200
    and random_reference_distribution_full['draw_index'].nunique() == 200
)

reload_verification_full = {
    'config_hash_matches': bool(config_hash_matches_full),
    'prediction_reload_matches_within_tolerance': bool(prediction_reload_matches_full),
    'row_identity_matches': bool(row_identity_matches_full),
    'random_scores_exact_match': bool(random_scores_exact_match_full),
    'random_reference_200_draws_self_consistent': bool(random_reference_self_consistent_full),
    'tolerance': {'rtol': 1e-6, 'atol': 1e-8},
}
print(reload_verification_full)
assert all(reload_verification_full[k] for k in (
    'config_hash_matches', 'prediction_reload_matches_within_tolerance',
    'row_identity_matches', 'random_scores_exact_match', 'random_reference_200_draws_self_consistent',
))


{'config_hash_matches': True, 'prediction_reload_matches_within_tolerance': True, 'row_identity_matches': True, 'random_scores_exact_match': True, 'random_reference_200_draws_self_consistent': True, 'tolerance': {'rtol': 1e-06, 'atol': 1e-08}}


In [27]:
full_wall_seconds = __import__('time').perf_counter() - full_wall_start
full_peak_rss_bytes = process.memory_info().rss

write_json_new(FULL_RUN_ROOT, 'audit/environment.json', {
    'run_id': FULL_RUN_ID,
    'created_at_utc': full_started.isoformat(),
    'git_head': full_git_head,
    'git_dirty': full_git_dirty,
    'python': sys.version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'lightgbm': lgb.__version__,
    'psutil': psutil.__version__,
    'total_ram_bytes': psutil.virtual_memory().total,
})

write_json_new(FULL_RUN_ROOT, 'audit/run_config.json', {
    'run_id': FULL_RUN_ID,
    'created_at_utc': full_started.isoformat(),
    'stage': 't07_full',
    'population': 'full_development_population',
    'git_head': full_git_head,
    'git_dirty': full_git_dirty,
    'real_or_held_out_data_accessed': False,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None,
    'src_lightgbm_baseline_sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'),
    'src_metrics_sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'),
    'processed_sha256': processed_sha256,
    'split_membership_sha256': observed_membership_hash,
    't05_run_id': t05_config['lifecycle_state_evidence']['authorizing_run_id'],
    't04_lifecycle_state': t04_config['lifecycle_state'],
    't06_authoritative_run_id': t07_config['input']['metrics_interface']['authoritative_t06_run_id'],
    't06_authoritative_definitions_sha256': t07_config['input']['metrics_interface']['authoritative_t06_definitions_sha256'],
    'scale_gating': {'policy': 'D30', 'stage': 'FULL', 'resource_gates': RESOURCE_GATES, 'preceding_smoke_run_id': RUN_ID},
    'train_count': int(len(train_ids_full)),
    'validation_count': int(len(validation_ids_full)),
    'lightgbm_config': lgb_baseline.FROZEN_BINARY_CONFIG,
    'lightgbm_config_hash': response_model_full.config_hash,
    'lightgbm_best_iteration': response_model_full.best_iteration,
    'lightgbm_num_boost_round_cap': lgb_baseline.NUM_BOOST_ROUND_CAP,
    'lightgbm_early_stopping_rounds': lgb_baseline.EARLY_STOPPING_ROUNDS,
    'response_predictions_sha256': response_predictions_sha256_full,
    'random_scores_sha256': random_scores_sha256_full,
    'definitions_sha256': definitions_sha256_full,
    'resource_evidence': {
        'wall_seconds': full_wall_seconds,
        'baseline_rss_bytes': full_baseline_rss_bytes,
        'peak_rss_bytes': full_peak_rss_bytes,
        'peak_rss_delta_bytes': full_peak_rss_bytes - full_baseline_rss_bytes,
    },
    'reload_verification': reload_verification_full,
})

print(f'FULL resource evidence: wall_seconds={full_wall_seconds:.2f}, peak_rss_delta_bytes={full_peak_rss_bytes - full_baseline_rss_bytes:,}')


FULL resource evidence: wall_seconds=1754.07, peak_rss_delta_bytes=-1,975,951,360


In [28]:
finalize_artifact_manifest(
    FULL_RUN_ROOT,
    run_id=FULL_RUN_ID,
    final_status='COMPLETED_T07_FULL_VERIFIED',
    created_at_utc=datetime.now(timezone.utc).isoformat(),
    stage='t07_full',
    population='full_development_population',
    external_artifacts=[
        {'path': 'kaggle/02_uplift_modeling.ipynb#sources', 'role': 'human_readable_protocol_source',
         'sha256': notebook_source_sha256(NOTEBOOK_PATH) if NOTEBOOK_PATH.is_file() else None, 'status': 'PASS'},
        {'path': 'src/lightgbm_baseline.py', 'role': 'reusable_t07_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'lightgbm_baseline.py'), 'status': 'PASS'},
        {'path': 'src/metrics.py', 'role': 'reusable_t06_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'metrics.py'), 'status': 'PASS'},
        {'path': f'outputs/runs/{RUN_ID}', 'role': 'preceding_smoke_run', 'sha256': None, 'status': 'PASS'},
    ],
)
print(f'FULL run finalized: {FULL_RUN_ID}')

immutable_write_refused_full = False
try:
    write_json_new(FULL_RUN_ROOT, 'audit/should_be_refused.json', {'x': 1})
except (FileExistsError, DataContractError):
    immutable_write_refused_full = True
except Exception:
    immutable_write_refused_full = False
print(f'Immutable-run write refusal verified: {immutable_write_refused_full}')
assert immutable_write_refused_full


FULL run finalized: t07_full_20260818T111446Z_593205
Immutable-run write refusal verified: True


In [29]:
full_summary = {
    'run_id': FULL_RUN_ID,
    'preceding_smoke_run_id': RUN_ID,
    'train_count': int(len(train_ids_full)),
    'validation_count': int(len(validation_ids_full)),
    'lightgbm_version': lgb.__version__,
    'lightgbm_config_hash': response_model_full.config_hash,
    'best_iteration': response_model_full.best_iteration,
    'response_diagnostics': response_diag_full.__dict__,
    'random_reference_draws': len(random_reference_distribution_full),
    'random_theoretical_qini_area': random_ranking_full.theoretical_random_qini_area,
    'random_illustrative_qini_above_random': random_ranking_full.qini_above_random,
    'response_qini_area': response_ranking_full.qini_area,
    'response_qini_above_random': response_ranking_full.qini_above_random,
    'response_uplift_at_k': response_ranking_full.uplift_at_k,
    'response_incremental_conversions_at_k': response_ranking_full.incremental_conversions_at_k,
    'ate': ate_full.__dict__,
    'reload_verification': reload_verification_full,
    'resource_wall_seconds': full_wall_seconds,
    'immutable_write_refused': immutable_write_refused_full,
}
print(json.dumps(full_summary, indent=2, default=str))


{
  "run_id": "t07_full_20260818T111446Z_593205",
  "preceding_smoke_run_id": "t07_smoke_20260818T111252Z_867633",
  "train_count": 9785714,
  "validation_count": 2096938,
  "lightgbm_version": "4.7.0",
  "lightgbm_config_hash": "647b625dca13ea63f61a8ded8ce97c5e4250beea1d09cc6fa1e3ef3bbb6eaf30",
  "best_iteration": 1,
  "response_diagnostics": {
    "roc_auc": 0.9038218277595335,
    "average_precision": 0.20332579606357337,
    "log_loss": 0.02076564385058888
  },
  "random_reference_draws": 200,
  "random_theoretical_qini_area": 1024.6686060004897,
  "random_illustrative_qini_above_random": -18.489185522843627,
  "response_qini_area": -1114.6438986351407,
  "response_qini_above_random": -2139.3125046356304,
  "response_uplift_at_k": {
    "10pct": 0.007330652775548674,
    "20pct": -0.006070526631827052,
    "30pct": -0.010081975358519447,
    "50pct": -0.001561196862742649,
    "100pct": 0.0011497647336709941
  },
  "response_incremental_conversions_at_k": {
    "10pct": 1537.193903

## 6. Interpretation

In [30]:
display(Markdown(
    f"Response's own diagnostics (ROC-AUC {response_diag_full.roc_auc:.4f}) describe how well "
    f"`P(Y=1|X)` predicts conversion -- a factual-outcome quality measure, never a causal ranking claim "
    f"(D12, D27). Whether that translates into a *useful uplift ranking* is a separate question, answered "
    f"only by routing the same scores through the frozen T06 Qini/uplift-at-K interface: Response reaches "
    f"`qini_above_random = {response_ranking_full.qini_above_random:.4f}` against a theoretical random "
    f"reference of {response_ranking_full.theoretical_random_qini_area:.4f}, and the illustrative seeded "
    f"random ranking reaches `qini_above_random = {random_ranking_full.qini_above_random:.4f}` of its own "
    f"-- both are read directly off the same metric interface with no separate formula for either method.\n\n"
    f"{'Response ranks above the random reference on this development population.' if response_ranking_full.qini_above_random > random_ranking_full.qini_above_random else 'Response does not clearly outrank the random reference on this development population -- a strong response model is not automatically a good uplift ranker, and that is a legitimate, reportable finding here, not an error.'} "
    f"This is a development-only observation on the validation partition; it is not a held-out claim, and "
    f"predicted uplift is not a true individual treatment effect. T-Learner, X-Learner, and Causal Forest "
    f"(T08-T11) extend this same comparison with genuinely causal estimators."
))


Response's own diagnostics (ROC-AUC 0.9038) describe how well `P(Y=1|X)` predicts conversion -- a factual-outcome quality measure, never a causal ranking claim (D12, D27). Whether that translates into a *useful uplift ranking* is a separate question, answered only by routing the same scores through the frozen T06 Qini/uplift-at-K interface: Response reaches `qini_above_random = -2139.3125` against a theoretical random reference of 1024.6686, and the illustrative seeded random ranking reaches `qini_above_random = -18.4892` of its own -- both are read directly off the same metric interface with no separate formula for either method.

Response does not clearly outrank the random reference on this development population -- a strong response model is not automatically a good uplift ranker, and that is a legitimate, reportable finding here, not an error. This is a development-only observation on the validation partition; it is not a held-out claim, and predicted uplift is not a true individual treatment effect. T-Learner, X-Learner, and Causal Forest (T08-T11) extend this same comparison with genuinely causal estimators.